# Peluang Ekspor Indonesia (BACI HS17)
Alur: data BACI → indikator & skor peluang → model gravity (potensi vs aktual).

## 1. Cek struktur data
Intip kolom file negara, produk, dan transaksi BACI.

In [1]:
# cek_kolom.py
import os
import pandas as pd

os.chdir(r"C:\Users\Bima Sena\Downloads\export - finder")

folder_baci = "data/BACI_HS17_V202601"

print(pd.read_csv(f"{folder_baci}/country_codes_V202601.csv").head(3))
print(pd.read_csv(f"{folder_baci}/product_codes_HS17_V202601.csv").head(3))
print(pd.read_csv(f"{folder_baci}/BACI_HS17_Y2018_V202601.csv", nrows=3))

   country_code country_name country_iso2 country_iso3
0             4  Afghanistan           AF          AFG
1             8      Albania           AL          ALB
2            12      Algeria           DZ          DZA
    code                                        description
0  10121           Horses: live, pure-bred breeding animals
1  10129  Horses: live, other than pure-bred breeding an...
2  10130                                        Asses: live
      t  i   j       k        v       q
0  2018  4  24  845420  115.180  26.000
1  2018  4  24  848180    2.632   0.007
2  2018  4  31  570110    1.596   0.037


## 2. Pengaturan
Folder, tahun (2018 & 2023), dan bab HS yang dipakai.

In [2]:
import os
import duckdb
import pandas as pd

os.chdir(r"C:\Users\Bima Sena\Downloads\export - finder")
os.makedirs("data/parquet", exist_ok=True)
os.makedirs("data/hasil", exist_ok=True)

con = duckdb.connect()

folder_baci = "data/BACI_HS17_V202601"
tahun_dipakai = [2018, 2023]
bab_pilihan = ["03", "09", "15", "18", "40", "44", "61", "62", "64"]
daftar_bab = ",".join(f"'{b}'" for b in bab_pilihan)

file_baci = [f"{folder_baci}/BACI_HS17_Y{t}_V202601.csv" for t in tahun_dipakai]
for f in file_baci:
    print(f, "ada:", os.path.exists(f))

data/BACI_HS17_V202601/BACI_HS17_Y2018_V202601.csv ada: True
data/BACI_HS17_V202601/BACI_HS17_Y2023_V202601.csv ada: True


## 3. Konversi BACI ke Parquet
Nilai diubah ke USD (BACI dalam ribu USD), kode negara jadi ISO3, hanya bab terpilih.

In [3]:
# kode negara
negara = pd.read_csv(f"{folder_baci}/country_codes_V202601.csv")
negara = negara.rename(columns={
    "country_code": "kode",
    "country_name": "nama",
    "country_iso3": "iso3",
})[["kode", "nama", "iso3"]]
negara.to_parquet("data/parquet/negara.parquet")

# kode produk
produk = pd.read_csv(f"{folder_baci}/product_codes_HS17_V202601.csv", dtype=str)
produk = produk.rename(columns={"code": "produk", "description": "deskripsi"})
produk["produk"] = produk["produk"].str.zfill(6)
produk.to_parquet("data/parquet/produk.parquet")

# baca csv baci (semua kolom dibaca sebagai teks dulu, lalu diubah ke angka)
con.register("negara", negara)
con.execute(f"""
COPY (
    SELECT b.tahun, e.iso3 AS eksportir, i.iso3 AS importir,
           b.produk, b.nilai_usd, b.berat_ton
    FROM (
        SELECT TRY_CAST(t AS INTEGER) AS tahun,
               TRY_CAST(i AS INTEGER) AS kode_eksportir,
               TRY_CAST(j AS INTEGER) AS kode_importir,
               lpad(trim(k), 6, '0') AS produk,
               TRY_CAST(v AS DOUBLE) * 1000 AS nilai_usd,
               TRY_CAST(q AS DOUBLE) AS berat_ton
        FROM read_csv({file_baci}, header=true, all_varchar=true)
    ) b
    JOIN negara e ON b.kode_eksportir = e.kode
    JOIN negara i ON b.kode_importir = i.kode
    WHERE substr(b.produk, 1, 2) IN ({daftar_bab})
      AND b.nilai_usd IS NOT NULL
) TO 'data/parquet/baci.parquet' (FORMAT PARQUET)
""")
print("selesai")

selesai


Cek cepat: total nilai per tahun dan ekspor Indonesia per bab.

In [4]:
# total per tahun
display(con.execute("""
    SELECT tahun, count(*) AS jumlah_baris, round(sum(nilai_usd) / 1e9, 1) AS total_miliar_usd
    FROM 'data/parquet/baci.parquet'
    GROUP BY tahun ORDER BY tahun
""").df())

# ekspor Indonesia per kelompok produk (HS 2 digit), tahun 2023
display(con.execute("""
    SELECT substr(produk, 1, 2) AS bab, round(sum(nilai_usd) / 1e9, 2) AS ekspor_idn_miliar_usd
    FROM 'data/parquet/baci.parquet'
    WHERE eksportir = 'IDN' AND tahun = 2023
    GROUP BY bab ORDER BY ekspor_idn_miliar_usd DESC
""").df())

,tahun,jumlah_baris,total_miliar_usd
0,2018,1557756,1276.0
1,2023,1736553,1446.6


,bab,ekspor_idn_miliar_usd
0,15,31.27
1,64,9.06
2,40,5.97
3,62,5.10
4,44,4.45
5,61,4.40
6,03,3.75
7,09,1.86
8,18,1.31


## 4. Hitung indikator
Impor & pertumbuhan (CAGR), ekspor Indonesia, konsentrasi pemasok (HHI), dan pangsa dunia.

In [5]:
baci = "data/parquet/baci.parquet"

tahun_awal = con.execute(f"SELECT min(tahun) FROM '{baci}'").fetchone()[0]
tahun_akhir = con.execute(f"SELECT max(tahun) FROM '{baci}'").fetchone()[0]
selang = tahun_akhir - tahun_awal
print("tahun awal:", tahun_awal, "| tahun akhir:", tahun_akhir)

batas_nilai_pasar = 1_000_000   # abaikan pasar di bawah 1 juta USD
batas_pangsa_dunia = 0.01       # produk yang pangsa dunia Indonesia minimal 1%

# permintaan: impor tiap negara per produk
impor = con.execute(f"""
    SELECT importir, produk,
           sum(CASE WHEN tahun = {tahun_akhir} THEN nilai_usd END) AS impor_akhir,
           sum(CASE WHEN tahun = {tahun_awal} THEN nilai_usd END) AS impor_awal,
           sum(CASE WHEN tahun = {tahun_akhir} THEN nilai_usd END)
             / NULLIF(sum(CASE WHEN tahun = {tahun_akhir} THEN berat_ton END) * 1000, 0) AS harga_kg
    FROM '{baci}'
    GROUP BY importir, produk
""").df()

impor = impor[(impor["importir"] != "IDN")
              & (impor["impor_akhir"] > batas_nilai_pasar)
              & (impor["impor_awal"] > 0)].copy()
impor["cagr"] = (impor["impor_akhir"] / impor["impor_awal"]) ** (1 / selang) - 1

# ekspor Indonesia ke tiap negara
ekspor_indo = con.execute(f"""
    SELECT importir, produk, sum(nilai_usd) AS ekspor_indo
    FROM '{baci}'
    WHERE eksportir = 'IDN' AND tahun = {tahun_akhir}
    GROUP BY importir, produk
""").df()

# konsentrasi pemasok (HHI)
hhi = con.execute(f"""
    WITH pangsa AS (
        SELECT importir, produk,
               nilai_usd / sum(nilai_usd) OVER (PARTITION BY importir, produk) AS p
        FROM '{baci}'
        WHERE tahun = {tahun_akhir} AND nilai_usd > 0
    )
    SELECT importir, produk, sum(p * p) AS hhi
    FROM pangsa
    GROUP BY importir, produk
""").df()

# pangsa Indonesia di ekspor dunia per produk
ekspor_dunia = con.execute(f"""
    SELECT produk,
           sum(nilai_usd) AS ekspor_dunia,
           sum(CASE WHEN eksportir = 'IDN' THEN nilai_usd END) AS ekspor_indo_total
    FROM '{baci}'
    WHERE tahun = {tahun_akhir}
    GROUP BY produk
""").df()
ekspor_dunia["pangsa_dunia"] = ekspor_dunia["ekspor_indo_total"] / ekspor_dunia["ekspor_dunia"]
pangsa_dunia = ekspor_dunia[["produk", "pangsa_dunia"]].dropna()

print("impor:", len(impor), "| hhi:", len(hhi), "| produk dengan pangsa dunia:", len(pangsa_dunia))

tahun awal: 2018 | tahun akhir: 2023
impor: 36508 | hhi: 118679 | produk dengan pangsa dunia: 692


## 5. Gabung dan beri skor
Skor 0–100 dari rata-rata lima komponen (bobot sama untuk sementara).

In [6]:
# kalau kernel sempat restart, dua baris ini tetap aman dijalankan
negara = pd.read_parquet("data/parquet/negara.parquet")
produk = pd.read_parquet("data/parquet/produk.parquet")

# gabung semua komponen
tabel = impor.merge(ekspor_indo, on=["importir", "produk"], how="left")
tabel = tabel.merge(hhi, on=["importir", "produk"], how="left")
tabel = tabel.merge(pangsa_dunia, on="produk", how="inner")
tabel["ekspor_indo"] = tabel["ekspor_indo"].fillna(0)
tabel["pangsa_indo"] = tabel["ekspor_indo"] / tabel["impor_akhir"]
tabel = tabel[tabel["pangsa_dunia"] >= batas_pangsa_dunia].copy()

# skor tiap komponen (peringkat persentil)
tabel["skor_ukuran"] = tabel["impor_akhir"].rank(pct=True)
tabel["skor_tumbuh"] = tabel["cagr"].rank(pct=True)
tabel["skor_persaingan"] = 1 - tabel["hhi"].rank(pct=True)
tabel["skor_ruang"] = 1 - tabel["pangsa_indo"].rank(pct=True)
tabel["skor_unggul"] = tabel["pangsa_dunia"].rank(pct=True)

# skor sementara, bobot sama rata (nanti bobotnya diatur user di dashboard)
kolom_skor = ["skor_ukuran", "skor_tumbuh", "skor_persaingan", "skor_ruang", "skor_unggul"]
tabel["skor"] = tabel[kolom_skor].mean(axis=1) * 100

# buang nama negara lama supaya iso3 tidak kembar
negara_bersih = negara.copy()
negara_bersih["lama"] = negara_bersih["nama"].str.contains(r"\(\.\.\.", regex=True)
negara_bersih = negara_bersih.sort_values("lama").drop_duplicates("iso3", keep="first")
nama_negara = negara_bersih[["iso3", "nama"]].rename(columns={"iso3": "importir", "nama": "negara"})

# tambah nama negara dan produk
tabel = tabel.merge(nama_negara, on="importir", how="left")
tabel = tabel.merge(produk, on="produk", how="left")
tabel["bab"] = tabel["produk"].str[:2]

tabel.to_parquet("data/hasil/indikator.parquet")
print("jumlah baris indikator:", len(tabel))
print("baris dobel:", tabel.duplicated(["importir", "produk"]).sum())

jumlah baris indikator: 19049
baris dobel: 0


## 6. Lihat hasil
Pasar terbesar, skor tertinggi, dan pasar yang sudah dimasuki Indonesia.

In [7]:
pd.set_option("display.max_colwidth", 60)

print("10 pasar terbesar:")
display(tabel.nlargest(10, "impor_akhir")[["negara", "produk", "deskripsi", "impor_akhir"]])

print("10 skor tertinggi:")
display(tabel.nlargest(10, "skor")[["negara", "produk", "deskripsi", "impor_akhir", "cagr", "pangsa_indo", "skor"]])

10 pasar terbesar:


,negara,produk,deskripsi,impor_akhir
6346,USA,401110,"Rubber: new pneumatic tyres, of a kind used on motor car...",1.067932e+10
12446,USA,611020,"Jerseys, pullovers, cardigans, waistcoats and similar ar...",6.698359e+09
1197,USA,611030,"Jerseys, pullovers, cardigans, waistcoats and similar ar...",6.493895e+09
18178,India,151110,"Vegetable oils: palm oil and its fractions, crude, not c...",6.459002e+09
156,China,030617,"Crustaceans: frozen, shrimps and prawns, excluding cold-...",5.602719e+09
8741,USA,640399,"Footwear: n.e.c. in heading no. 6403, (not covering the ...",5.508333e+09
17763,USA,090111,Coffee: not roasted or decaffeinated,5.160450e+09
5476,Germany,401110,"Rubber: new pneumatic tyres, of a kind used on motor car...",4.879115e+09
16986,USA,030617,"Crustaceans: frozen, shrimps and prawns, excluding cold-...",4.831119e+09
1721,USA,620342,"Trousers, bib and brace overalls, breeches and shorts: m...",4.803519e+09


10 skor tertinggi:


,negara,produk,deskripsi,impor_akhir,cagr,pangsa_indo,skor
10921,Mexico,151110,"Vegetable oils: palm oil and its fractions, crude, not c...",486866764.0,0.134488,0.0,87.316395
10612,Cameroon,030389,"Fish: frozen, n.e.c. in heading 0303, excluding fillets,...",108596570.0,0.251295,0.0,85.880624
6251,Slovakia,151620,Vegetable fats and oils and their fractions: partly or w...,68477384.0,0.208810,0.0,85.491102
530,Cameroon,151110,"Vegetable oils: palm oil and its fractions, crude, not c...",157106032.0,0.568409,0.0,84.391307
14320,Congo,030389,"Fish: frozen, n.e.c. in heading 0303, excluding fillets,...",25163221.0,0.241622,0.0,84.176597
2929,Benin,030389,"Fish: frozen, n.e.c. in heading 0303, excluding fillets,...",19937083.0,0.473796,0.0,83.943514
6389,Austria,151800,Animal or vegetable fats and oils and their fractions: o...,197433491.0,0.215400,0.0,83.937740
1393,Germany,441234,"Plywood: consisting only of sheets of wood (not bamboo),...",17098485.0,0.201295,0.0,82.431624
335,Denmark,440122,"Wood: for fuel, in chips or particles, non-coniferous, w...",125480567.0,0.330179,0.0,82.204315
14727,Morocco,030743,"Molluscs: cuttle fish and squid, whether in shell or not...",89933258.0,0.218389,0.0,80.747546


In [8]:
sudah_ada = tabel[tabel["pangsa_indo"] > 0].copy()
sudah_ada["cagr_persen"] = (sudah_ada["cagr"] * 100).round(1)
sudah_ada["pangsa_persen"] = (sudah_ada["pangsa_indo"] * 100).round(2)

print("10 skor tertinggi di pasar yang sudah dimasuki:")
display(sudah_ada.nlargest(10, "skor")[
    ["negara", "produk", "deskripsi", "impor_akhir", "cagr_persen", "pangsa_persen", "skor"]
])

print("Jumlah peluang per kelompok produk:")
display(tabel["bab"].value_counts().sort_index())

10 skor tertinggi di pasar yang sudah dimasuki:


,negara,produk,deskripsi,impor_akhir,cagr_persen,pangsa_persen,skor
2355,Poland,620312,"Suits: men's or boys', of synthetic fibres (not knitted ...",32529424.0,39.8,0.07,80.613156
12053,Netherlands,151790,Edible mixtures or preparations of animal or vegetable f...,319185174.0,10.5,0.00,80.094493
9794,Qatar,620312,"Suits: men's or boys', of synthetic fibres (not knitted ...",25052255.0,83.1,0.06,79.963253
3030,Switzerland,151790,Edible mixtures or preparations of animal or vegetable f...,24132438.0,12.3,0.00,79.885558
13962,Rep. of Korea,030487,"Fish fillets: frozen, tunas (of the genus Thunnus), skip...",224270163.0,12.3,0.04,79.566381
2324,Germany,620312,"Suits: men's or boys', of synthetic fibres (not knitted ...",75561159.0,11.7,0.01,79.499186
11415,France,152200,Degras: residues resulting from the treatment of fatty s...,55651725.0,21.0,1.10,79.324374
16507,Bulgaria,640399,"Footwear: n.e.c. in heading no. 6403, (not covering the ...",70299532.0,14.7,0.01,79.163211
16342,Netherlands,621030,"Garments: overcoats, raincoats, car-coats, capes, cloaks...",127827253.0,66.7,0.27,78.800987
11872,Belgium,151800,Animal or vegetable fats and oils and their fractions: o...,426286999.0,41.4,0.24,78.733792


Jumlah peluang per kelompok produk:


bab
03    1825
09     970
15    1289
18     273
40    1784
44    2473
61    4312
62    4676
64    1447
Name: count, dtype: int64

## 7. Data gravity (CEPII)
Jarak, perbatasan, bahasa, koloni, dan FTA. Dipakai tahun 2020 (data terbaru di file).

In [9]:
file_grav = "data/Gravity_csv_V202211/Gravity_V202211.csv"
print("file ada:", os.path.exists(file_grav))

contoh = pd.read_csv(file_grav, nrows=3)
kolom_perlu = ["year", "iso3_o", "iso3_d", "dist", "contig", "comlang_off", "comcol", "fta_wto"]
print("kolom yang belum ketemu:", [k for k in kolom_perlu if k not in contoh.columns])
print(list(contoh.columns))

file ada: True
kolom yang belum ketemu: []
['year', 'country_id_o', 'country_id_d', 'iso3_o', 'iso3_d', 'iso3num_o', 'iso3num_d', 'country_exists_o', 'country_exists_d', 'gmt_offset_2020_o', 'gmt_offset_2020_d', 'distw_harmonic', 'distw_arithmetic', 'distw_harmonic_jh', 'distw_arithmetic_jh', 'dist', 'main_city_source_o', 'main_city_source_d', 'distcap', 'contig', 'diplo_disagreement', 'scaled_sci_2021', 'comlang_off', 'comlang_ethno', 'comcol', 'col45', 'legal_old_o', 'legal_old_d', 'legal_new_o', 'legal_new_d', 'comleg_pretrans', 'comleg_posttrans', 'transition_legalchange', 'comrelig', 'heg_o', 'heg_d', 'col_dep_ever', 'col_dep', 'col_dep_end_year', 'col_dep_end_conflict', 'empire', 'sibling_ever', 'sibling', 'sever_year', 'sib_conflict', 'pop_o', 'pop_d', 'gdp_o', 'gdp_d', 'gdpcap_o', 'gdpcap_d', 'pop_source_o', 'pop_source_d', 'gdp_source_o', 'gdp_source_d', 'gdp_ppp_o', 'gdp_ppp_d', 'gdpcap_ppp_o', 'gdpcap_ppp_d', 'pop_pwt_o', 'pop_pwt_d', 'gdp_ppp_pwt_o', 'gdp_ppp_pwt_d', 'gatt_

In [10]:
con.execute(f"""
COPY (
    SELECT iso3_o AS eksportir, iso3_d AS importir,
           TRY_CAST(dist AS DOUBLE) AS dist,
           COALESCE(TRY_CAST(contig AS DOUBLE), 0) AS contig,
           COALESCE(TRY_CAST(comlang_off AS DOUBLE), 0) AS comlang_off,
           COALESCE(TRY_CAST(comcol AS DOUBLE), 0) AS comcol,
           COALESCE(TRY_CAST(fta_wto AS DOUBLE), 0) AS fta_wto
    FROM read_csv('{file_grav}', header=true, all_varchar=true)
    WHERE year = '2020'
) TO 'data/parquet/gravity.parquet' (FORMAT PARQUET)
""")

grav = pd.read_parquet("data/parquet/gravity.parquet")
print(len(grav), "pasangan negara")
display(grav.head(3))

63504 pasangan negara


,eksportir,importir,dist,contig,comlang_off,comcol,fta_wto
0,ABW,ABW,5.0,0.0,0.0,0.0,0.0
1,ABW,AFG,13256.0,0.0,0.0,0.0,0.0
2,ABW,AGO,9505.0,0.0,0.0,0.0,0.0


## 8. GDP World Bank
GDP (USD berjalan) 2015–2024 lewat API.

In [11]:
import requests

def ambil_wb(kode_indikator, nama_kolom):
    url = f"https://api.worldbank.org/v2/country/all/indicator/{kode_indikator}"
    param = {"format": "json", "per_page": 20000, "date": "2015:2024"}
    isi = requests.get(url, params=param, timeout=60).json()[1]
    tabel_wb = pd.DataFrame([
        {"iso3": x["countryiso3code"], "tahun": int(x["date"]), nama_kolom: x["value"]}
        for x in isi
    ])
    return tabel_wb[tabel_wb["iso3"] != ""]

gdp = ambil_wb("NY.GDP.MKTP.CD", "gdp")
gdp.to_parquet("data/parquet/gdp.parquet")

# jumlah negara yang punya data GDP per tahun
print(gdp.dropna(subset=["gdp"]).groupby("tahun")["gdp"].count())

tahun
2015    256
2016    255
2017    255
2018    255
2019    255
2020    254
2021    254
2022    253
2023    248
2024    244
Name: gdp, dtype: int64


## 9. Model gravity (PPML)
Regresi Poisson (PPML) pada semua pasangan negara, nol ikut dipakai. Potensi Indonesia dikalibrasi ke total ekspornya sendiri, jadi selisih positif = pasar yang kurang tergarap dibanding pola ekspor Indonesia.

In [12]:
tahun_model = 2023   # harus salah satu tahun yang ada di baci.parquet

grav = pd.read_parquet("data/parquet/gravity.parquet")
gdp = pd.read_parquet("data/parquet/gdp.parquet")

# Ambil GDP terbaru yang TERSEDIA (<= tahun_model). Baris dengan nilai kosong dibuang dulu,
# supaya negara yang kosong di tahun itu (mis. IDN) tidak lolos merge dengan GDP = NaN.
gdp_tahun = (gdp.dropna(subset=["gdp"])
                .query("tahun <= @tahun_model")
                .sort_values("tahun")
                .drop_duplicates("iso3", keep="last")[["iso3", "gdp"]])

gdp_e = gdp_tahun.rename(columns={"iso3": "eksportir", "gdp": "gdp_e"})
gdp_i = gdp_tahun.rename(columns={"iso3": "importir", "gdp": "gdp_i"})

print("GDP IDN:", gdp_tahun.loc[gdp_tahun["iso3"] == "IDN", "gdp"].values)

GDP IDN: [1.37116693e+12]


In [ ]:
import numpy as np

def ppml(X, y, iters=100, tol=1e-9):
    """Poisson pseudo-maximum likelihood (IRLS). Cocok untuk aliran dagang: nol ikut dipakai, tidak bias seperti OLS pada log."""
    beta = np.zeros(X.shape[1])
    beta[0] = np.log(y.mean() + 1e-9)
    for _ in range(iters):
        eta = np.clip(X @ beta, -30, 30)
        mu = np.exp(eta)
        z = eta + (y - mu) / mu
        XtW = X.T * mu
        beta_baru = np.linalg.solve(XtW @ X + 1e-8 * np.eye(X.shape[1]), XtW @ z)
        selesai = np.max(np.abs(beta_baru - beta)) < tol
        beta = beta_baru
        if selesai:
            break
    return beta

fitur = ["ln_gdp_e", "ln_gdp_i", "ln_dist", "contig", "comlang_off", "comcol", "fta_wto"]
fitur_log = ["ln_gdp_e", "ln_gdp_i", "ln_dist"]
koefisien = {}
hasil_semua = []
cek_idn = {}

for bab in bab_pilihan:
    pasangan = con.execute(f"""
        SELECT eksportir, importir, sum(nilai_usd) / 1e6 AS nilai
        FROM 'data/parquet/baci.parquet'
        WHERE tahun = {tahun_model} AND substr(produk, 1, 2) = '{bab}'
        GROUP BY eksportir, importir
    """).df()
    penjual = pasangan["eksportir"].unique()

    data = grav[grav["eksportir"].isin(penjual)]
    data = data.merge(pasangan, on=["eksportir", "importir"], how="left")
    data = data.merge(gdp_e, on="eksportir").merge(gdp_i, on="importir")
    data = data[(data["eksportir"] != data["importir"])
                & (data["dist"] > 0) & (data["gdp_e"] > 0) & (data["gdp_i"] > 0)].copy()
    data["nilai"] = data["nilai"].fillna(0.0)          # pasangan tanpa perdagangan = nol sungguhan
    data["ln_gdp_e"] = np.log(data["gdp_e"])
    data["ln_gdp_i"] = np.log(data["gdp_i"])
    data["ln_dist"] = np.log(data["dist"])
    cek_idn[bab] = int((data["eksportir"] == "IDN").sum())

    # variabel log dipusatkan supaya perhitungan stabil
    rata = data[fitur_log].mean()
    data[fitur_log] = data[fitur_log] - rata

    X = np.column_stack([np.ones(len(data)), data[fitur].to_numpy(float)])
    beta = ppml(X, data["nilai"].to_numpy(float))
    koefisien[bab] = dict(zip(["const"] + fitur, beta))

    idn = data[data["eksportir"] == "IDN"].copy()
    Xi = np.column_stack([np.ones(len(idn)), idn[fitur].to_numpy(float)])
    idn["potensi_mentah"] = np.exp(Xi @ beta)
    idn["aktual"] = idn["nilai"]

    # kalibrasi ke total ekspor Indonesia sendiri: model gravity tidak tahu kapasitas produksi Indonesia,
    # jadi "potensi" dibaca sebagai pembagian ekspor Indonesia yang wajar menurut GDP, jarak, bahasa, FTA
    idn["potensi"] = idn["potensi_mentah"] * idn["aktual"].sum() / idn["potensi_mentah"].sum()
    idn["selisih"] = idn["potensi"] - idn["aktual"]
    idn["bab"] = bab
    hasil_semua.append(idn[["bab", "importir", "aktual", "potensi_mentah", "potensi", "selisih"]])

print("baris IDN yang masuk model per bab:", cek_idn)
assert all(v > 0 for v in cek_idn.values()), "IDN hilang dari data model, cek GDP/dist IDN"


## 10. Hasil gravity
Gabung nama negara, simpan, dan lihat peluang terbesar (juta USD).

In [ ]:
hasil = pd.concat(hasil_semua)

# nama negara (tanpa nama lama yang kembar)
negara_bersih = negara.copy()
negara_bersih["lama"] = negara_bersih["nama"].str.contains(r"\(\.\.\.", regex=True)
negara_bersih = negara_bersih.sort_values("lama").drop_duplicates("iso3", keep="first")
nama_negara = negara_bersih[["iso3", "nama"]].rename(columns={"iso3": "importir", "nama": "negara"})

hasil = hasil.merge(nama_negara, on="importir", how="left")
hasil.to_parquet("data/hasil/gravity.parquet")
print("baris hasil gravity:", len(hasil))

print("Koefisien model per kelompok produk:")
display(pd.DataFrame(koefisien).T)

for b in ["15", "09"]:
    print("Bab", b, ": potensi dikurangi aktual terbesar (juta USD)")
    display(hasil[hasil["bab"] == b].nlargest(10, "selisih")[["negara", "aktual", "potensi", "selisih"]].round(1))

## 11. Tahap 5: Forecasting permintaan impor
Proses semua tahun BACI (bukan hanya 2018 & 2023) agar ada deret waktu.

In [15]:
import glob, re
import numpy as np

# semua tahun BACI yang ada di folder (bukan cuma 2018 & 2023)
file_semua = sorted(f.replace("\\", "/") for f in glob.glob(f"{folder_baci}/BACI_HS17_Y*_V202601.csv"))
tahun_semua = [int(re.search(r"_Y(\d{4})_", f).group(1)) for f in file_semua]
print("tahun tersedia:", tahun_semua)

negara = pd.read_parquet("data/parquet/negara.parquet")
con.register("negara", negara)

# cukup total impor per importir-produk-tahun (permintaan), jadi file kecil
con.execute(f"""
COPY (
    SELECT b.tahun, i.iso3 AS importir, b.produk, sum(b.nilai_usd) AS nilai_usd
    FROM (
        SELECT TRY_CAST(t AS INTEGER) AS tahun,
               TRY_CAST(j AS INTEGER) AS kode_importir,
               lpad(trim(k), 6, '0') AS produk,
               TRY_CAST(v AS DOUBLE) * 1000 AS nilai_usd
        FROM read_csv({file_semua}, header=true, all_varchar=true)
    ) b
    JOIN negara i ON b.kode_importir = i.kode
    WHERE substr(b.produk, 1, 2) IN ({daftar_bab}) AND b.nilai_usd IS NOT NULL
    GROUP BY b.tahun, i.iso3, b.produk
) TO 'data/parquet/impor_tahunan.parquet' (FORMAT PARQUET)
""")
print("selesai")

tahun tersedia: [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
selesai


Susun deret tahunan impor per negara-produk. Hanya seri lengkap dan bernilai > 0 yang diramal.

In [16]:
impor_th = pd.read_parquet("data/parquet/impor_tahunan.parquet")
tahun_semua = sorted(impor_th["tahun"].unique())

mat = impor_th.pivot_table(index=["importir", "produk"], columns="tahun", values="nilai_usd")

# hanya seri yang ada di tabel skor, dengan data lengkap dan > 0 di semua tahun
kunci = pd.MultiIndex.from_frame(tabel[["importir", "produk"]])
mat = mat[mat.index.isin(kunci)].dropna()
mat = mat[(mat > 0).all(axis=1)]
print("seri di tabel skor:", len(kunci), "| seri lengkap yang bisa diramal:", len(mat))

seri di tabel skor: 19049 | seri lengkap yang bisa diramal: 18923


Backtest: tren log-linear vs naif (nilai terakhir) vs tren diredam. Metode dengan galat median terkecil dipakai.

In [17]:
def fit_tren(Y, x):
    """Regresi linear pada log(nilai). Y: (jumlah_seri, jumlah_tahun)."""
    L = np.log(Y)
    xc = x - x.mean()
    rata = L.mean(axis=1, keepdims=True)
    slope = (L - rata) @ xc / (xc @ xc)
    level = rata[:, 0] + slope * (x[-1] - x.mean())        # nilai garis tren di tahun terakhir
    resid = L - (rata + np.outer(slope, xc))
    sigma = np.sqrt((resid ** 2).sum(axis=1) / (len(x) - 2))
    return level, slope, sigma

def prediksi(Y, x, h, metode):
    """Prediksi untuk langkah ke depan h (array, 0 = tahun terakhir)."""
    if metode == "naif":
        return np.repeat(Y[:, [-1]], len(h), axis=1)
    level, slope, _ = fit_tren(Y, x)
    redam = {"tren": 1.0, "tren_redam": 0.5}[metode]
    return np.exp(level[:, None] + redam * slope[:, None] * h[None, :])

x = np.array(tahun_semua, float)
Y = mat.to_numpy(float)

# backtest: latih tanpa 2 tahun terakhir, tebak 2 tahun itu, bandingkan dengan aktual
k = 2
bt = pd.DataFrame(index=mat.index)
for m in ["naif", "tren", "tren_redam"]:
    pred = prediksi(Y[:, :-k], x[:-k], np.arange(1, k + 1), m)
    bt[m] = np.abs(np.log(pred / Y[:, -k:])).mean(axis=1)

print("median galat (|ln prediksi/aktual|, 0.2 kira-kira meleset 22%):")
print(bt.median().round(3))
metode_terbaik = bt.median().idxmin()
print("metode terpilih:", metode_terbaik)

bt["bab"] = bt.index.get_level_values("produk").str[:2]
display(bt.groupby("bab")[["naif", "tren", "tren_redam"]].median().round(3))

median galat (|ln prediksi/aktual|, 0.2 kira-kira meleset 22%):
naif          0.269
tren          0.354
tren_redam    0.322
dtype: float64
metode terpilih: naif


,naif,tren,tren_redam
bab,,,
03,0.283,0.379,0.344
09,0.265,0.332,0.309
15,0.333,0.417,0.353
18,0.391,0.353,0.363
40,0.262,0.315,0.286
44,0.295,0.374,0.331
61,0.259,0.353,0.310
62,0.246,0.346,0.326
64,0.258,0.334,0.309


Ramal 3 tahun ke depan dengan selang ~80%, gabung ke tabel skor, dan hitung skor versi proyeksi.

In [18]:
# CATATAN: backtest memilih metode naif, jadi cagr_proyeksi = 0 dan skor_proyeksi tidak dipakai.
# Skor yang dipakai dashboard adalah skor_v2 (sel 12). Sel ini tetap dijalankan karena menyimpan forecast.parquet.
horizon = 3
h = np.arange(1, horizon + 1)
tahun_terakhir = int(x[-1])
tahun_target = tahun_terakhir + horizon

pred = prediksi(Y, x, h, metode_terbaik)
dasar = prediksi(Y, x, np.array([0]), metode_terbaik)[:, 0]
_, _, sigma = fit_tren(Y, x)
lebar = np.exp(1.28 * sigma * np.sqrt(horizon))             # selang kira-kira 80%

fc = pd.DataFrame({
    "impor_proyeksi": pred[:, -1],
    "batas_bawah": pred[:, -1] / lebar,
    "batas_atas": pred[:, -1] * lebar,
    "cagr_proyeksi": (pred[:, -1] / dasar) ** (1 / horizon) - 1,
}, index=mat.index).reset_index()
fc["tahun_proyeksi"] = tahun_target
fc.to_parquet("data/hasil/forecast.parquet")

# masukkan ke tabel skor (aman dijalankan ulang)
kolom_fc = ["impor_proyeksi", "batas_bawah", "batas_atas", "cagr_proyeksi", "tahun_proyeksi"]
tabel = tabel.drop(columns=[c for c in kolom_fc if c in tabel.columns])
tabel = tabel.merge(fc, on=["importir", "produk"], how="left")

# skor versi proyeksi: komponen pertumbuhan pakai CAGR proyeksi, bukan CAGR 2018-2023
ada_fc = tabel["cagr_proyeksi"].notna()
tabel["skor_tumbuh_proyeksi"] = tabel.loc[ada_fc, "cagr_proyeksi"].rank(pct=True)
kolom_v2 = ["skor_ukuran", "skor_tumbuh_proyeksi", "skor_persaingan", "skor_ruang", "skor_unggul"]
tabel["skor_proyeksi"] = tabel[kolom_v2].mean(axis=1) * 100
tabel.loc[~ada_fc, "skor_proyeksi"] = np.nan

tabel.to_parquet("data/hasil/indikator_forecast.parquet")
print(f"proyeksi {tahun_target} untuk {ada_fc.sum()} dari {len(tabel)} baris tabel")

tampil = tabel[ada_fc].nlargest(10, "skor_proyeksi").copy()
tampil["cagr_proyeksi_%"] = (tampil["cagr_proyeksi"] * 100).round(1)
display(tampil[["negara", "produk", "deskripsi", "impor_terakhir" if "impor_terakhir" in tampil else "impor_akhir",
                "impor_proyeksi", "cagr_proyeksi_%", "skor_proyeksi"]])

proyeksi 2027 untuk 18923 dari 19049 baris tabel


,negara,produk,deskripsi,impor_akhir,impor_proyeksi,cagr_proyeksi_%,skor_proyeksi
10921,Mexico,151110,"Vegetable oils: palm oil and its fractions, crude, not c...",486866764.0,466927709.0,0.0,82.642137
3814,Germany,152000,"Glycerol, crude: glycerol waters and glycerol lyes",41438359.0,47141118.0,0.0,80.314456
11313,Hungary,640391,"Footwear: n.e.c. in heading no. 6403, covering the ankle...",86814532.0,79129350.0,0.0,79.482916
6251,Slovakia,151620,Vegetable fats and oils and their fractions: partly or w...,68477384.0,55660925.0,0.0,78.908083
10612,Cameroon,030389,"Fish: frozen, n.e.c. in heading 0303, excluding fillets,...",108596570.0,53873977.0,0.0,78.640352
274,TÃ¼rkiye,400219,"Rubber: synthetic, styrene-butadiene rubber (SBR) and ca...",236485285.0,242649776.0,0.0,78.559508
3681,Italy,030289,"Fish: fresh or chilled, n.e.c. in heading 0302, excludin...",137040111.0,171222381.0,0.0,78.415143
8789,Germany,441233,Plywood: with sheets of wood only: not bamboo: each ply ...,504959185.0,454437572.0,0.0,78.120115
15554,Belgium,400219,"Rubber: synthetic, styrene-butadiene rubber (SBR) and ca...",93410764.0,112742149.0,0.0,78.050295
5357,Austria,441294,"Blockboard, laminboard and battenboard (not bamboo, and ...",65262571.0,60147193.0,0.0,77.734268


## 12. Perbaikan nama negara & skor pertumbuhan
Backtest memilih metode naif (growth = 0), jadi pertumbuhan diukur dari laju tren log 2017–2024. Hasil disimpan untuk dashboard.

In [19]:
# --- perbaiki nama negara (mojibake: "TÃ¼rkiye" -> "Türkiye") ---
def perbaiki(s):
    if not isinstance(s, str):      # NaN / kosong dilewati
        return s
    try:
        return s.encode("latin-1").decode("utf-8")
    except (UnicodeEncodeError, UnicodeDecodeError):
        return s

tabel["negara"] = tabel["negara"].map(perbaiki).fillna(tabel["importir"])

# --- skor pertumbuhan baru ---
# Backtest memilih "naif", jadi cagr_proyeksi = 0 untuk semua baris dan tidak bisa dipakai meranking.
# Sebagai gantinya: laju tren log 2017-2024 (lebih stabil daripada CAGR dua titik) + ukuran pasar tahun terakhir.
level, slope, sigma = fit_tren(Y, x)
tren = pd.DataFrame({
    "impor_terakhir": Y[:, -1],
    "cagr_tren": np.exp(slope) - 1,
}, index=mat.index).reset_index()

tabel = tabel.drop(columns=[c for c in ["impor_terakhir", "cagr_tren"] if c in tabel.columns])
tabel = tabel.merge(tren, on=["importir", "produk"], how="left")

ada = tabel["cagr_tren"].notna()
tabel["skor_ukuran_v2"] = tabel.loc[ada, "impor_terakhir"].rank(pct=True)
tabel["skor_tumbuh_v2"] = tabel.loc[ada, "cagr_tren"].rank(pct=True)

kolom_v2 = ["skor_ukuran_v2", "skor_tumbuh_v2", "skor_persaingan", "skor_ruang", "skor_unggul"]
tabel["skor_v2"] = tabel[kolom_v2].mean(axis=1) * 100
tabel.loc[~ada, "skor_v2"] = np.nan

tabel.to_parquet("data/hasil/indikator_final.parquet")

# nama negara di hasil gravity juga diperbaiki
hasil["negara"] = hasil["negara"].map(perbaiki).fillna(hasil["importir"])
hasil.to_parquet("data/hasil/gravity.parquet")

tampil = tabel[ada].nlargest(10, "skor_v2").copy()
tampil["cagr_tren_%"] = (tampil["cagr_tren"] * 100).round(1)
display(tampil[["negara", "produk", "deskripsi", "impor_terakhir", "cagr_tren_%", "skor_v2"]])

,negara,produk,deskripsi,impor_terakhir,cagr_tren_%,skor_v2
10921,Mexico,151110,"Vegetable oils: palm oil and its fractions, crude, not c...",466927709.0,11.0,86.716441
9756,Lithuania,030617,"Crustaceans: frozen, shrimps and prawns, excluding cold-...",58704069.0,29.7,85.909888
6251,Slovakia,151620,Vegetable fats and oils and their fractions: partly or w...,55660925.0,18.8,85.157310
10612,Cameroon,030389,"Fish: frozen, n.e.c. in heading 0303, excluding fillets,...",53873977.0,22.5,84.857236
6389,Austria,151800,Animal or vegetable fats and oils and their fractions: o...,134455804.0,15.8,82.764276
2929,Benin,030389,"Fish: frozen, n.e.c. in heading 0303, excluding fillets,...",17821198.0,23.8,82.392674
14320,Congo,030389,"Fish: frozen, n.e.c. in heading 0303, excluding fillets,...",13069058.0,18.8,81.608789
16195,Netherlands,180320,"Cocoa: paste, wholly or partly defatted",229244365.0,34.9,81.566776
2355,Poland,620312,"Suits: men's or boys', of synthetic fibres (not knitted ...",48668342.0,30.5,81.426870
6617,Sweden,151620,Vegetable fats and oils and their fractions: partly or w...,36929183.0,11.2,81.230573


## 13. Data historis untuk dashboard
Menyimpan deret impor per tahun supaya tab Prakiraan bisa menggambar garis sejarah.

In [ ]:
# deret impor 2017-2024 untuk tab Prakiraan di dashboard (hanya seri yang ada di tabel skor, jadi file kecil)
impor_th = pd.read_parquet("data/parquet/impor_tahunan.parquet")
kunci = tabel[["importir", "produk"]].drop_duplicates()
deret = impor_th.merge(kunci, on=["importir", "produk"])
deret["nilai_usd"] = deret["nilai_usd"].astype("float32")
deret.to_parquet("data/hasil/deret_impor.parquet")
print("baris deret:", len(deret), "| seri:", len(kunci))

## 14. Ringkasan untuk dashboard
Menyimpan tahun data, hasil uji mundur, dan koefisien gravity agar halaman Ringkasan dan Model gravity terisi.

In [ ]:
# ringkasan untuk halaman "Ringkasan dan metode" di dashboard
import json

ringkasan = {
    "tahun_data": [int(t) for t in tahun_semua],
    "tahun_model_gravity": int(tahun_model),
    "jumlah_seri_skor": int(len(tabel)),
    "jumlah_seri_forecast": int(len(mat)),
    "horizon_tahun": int(horizon),
    "metode_forecast_terpilih": str(metode_terbaik),
    "backtest_global": {k: float(v) for k, v in bt[["naif", "tren", "tren_redam"]].median().items()},
    "backtest_per_bab": bt.groupby("bab")[["naif", "tren", "tren_redam"]].median().round(4).to_dict(orient="index"),
    "koefisien": {b: {k: float(v) for k, v in d.items()} for b, d in koefisien.items()},
}
with open("data/hasil/ringkasan.json", "w", encoding="utf-8") as f:
    json.dump(ringkasan, f, ensure_ascii=False, indent=2)
print("tersimpan:", list(ringkasan))